In [ ]:
import pickle
import os
import json
import math
import umap 
import torch
import random
import hashlib
import argparse

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
from copy import deepcopy
import scipy.sparse as sp
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch_scatter import scatter_mean
import torch.optim.lr_scheduler as lr_scheduler

from sklearn.manifold import TSNE
from scipy.sparse import coo_matrix
from collections import defaultdict
from torch_scatter import scatter_sum
from torch_geometric.utils import softmax
from sklearn.metrics import silhouette_score
from torch.utils.data import Dataset,DataLoader
from torch_geometric.data import Data, Batch
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch_geometric.utils import dense_to_sparse
from torch_geometric.nn import GCNConv, GATConv,GraphNorm,global_mean_pool
from sklearn.metrics import roc_auc_score,average_precision_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
)

import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print('torch version: ', torch.__version__)
torch.cuda.set_device(1)
print(device)

torch.set_printoptions(threshold=float('inf'), edgeitems=100, linewidth=200)

In [ ]:
def load_adj_matrices(folder_path, drop_suffix=True):
    adj_dict = {}
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):  # 只处理csv文件
            file_path = os.path.join(folder_path, filename)
            adj_matrix = pd.read_csv(file_path, index_col=0)  # 如果第一列是索引
            #adj_matrix = pd.read_csv(file_path, header=None)  # 如果没有表头
            key = os.path.splitext(filename)[0] if drop_suffix else filename
            
            #adj_dict[key] = adj_matrix.values  # 存为 numpy 数组
            adj_dict[key] = adj_matrix       # 如果想保留 DataFrame
            
    return adj_dict

def build_pathway_dict(folder_path, cnv_amp_file, cnv_del_file, snv_file):
    
    cnv_amp = pd.read_csv(cnv_amp_file, index_col=0)
    cnv_del = pd.read_csv(cnv_del_file, index_col=0)
    snv = pd.read_csv(snv_file, index_col=0)

    pathway_dict = {}

    for filename in os.listdir(folder_path):
        if not filename.endswith(".csv"):
            continue
        #print(filename)
        file_path = os.path.join(folder_path, filename)
        adj_matrix = pd.read_csv(file_path, index_col=0)
        genes_in_pathway = adj_matrix.index.tolist()

        sample_dict = {}
        for sample in cnv_amp.columns:  # 
            cnv_amp_vals = cnv_amp[sample].reindex(genes_in_pathway, fill_value=0)
            cnv_del_vals = cnv_del[sample].reindex(genes_in_pathway, fill_value=0) if sample in cnv_del.columns else pd.Series(0, index=genes_in_pathway)
            snv_vals = snv[sample].reindex(genes_in_pathway, fill_value=0) if sample in snv.columns else pd.Series(0, index=genes_in_pathway)
            df = pd.DataFrame({
                "CNV_amp": cnv_amp_vals,
                "CNV_del": cnv_del_vals,
                "SNV": snv_vals,
            })
            sample_dict[sample] = df
        key = os.path.splitext(filename)[0]
        pathway_dict[key] = sample_dict

    return pathway_dict

def build_pathway_dict_singleomic(folder_path, exp_file):

    exp_data = pd.read_csv(exp_file, index_col=0)

    pathway_dict = {}

    for filename in os.listdir(folder_path):
        if not filename.endswith(".csv"):
            continue
        #print(filename)
        file_path = os.path.join(folder_path, filename)
        adj_matrix = pd.read_csv(file_path, index_col=0)
        genes_in_pathway = adj_matrix.index.tolist()

        sample_dict = {}
        for sample in exp_data.columns:  # 
            exp_vals = exp_data[sample].reindex(genes_in_pathway, fill_value=0)
            df = pd.DataFrame({
                "Exp": exp_vals
            })
            sample_dict[sample] = df
        key = os.path.splitext(filename)[0]
        pathway_dict[key] = sample_dict

    return pathway_dict

def dot_product_decode(Z):
    return torch.sigmoid(torch.mm(Z, Z.t()))

def seed_everything(seed = 3078):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def save_exp_result(setting, result, dir_path):
    ''' Save result dictionaries as JSON file'''
    exp_name = setting['exp_name']
    #del setting['max_epoch']
    #del setting['train_batch_size']
    #del setting['test_batch_size']

    hash_key = hashlib.sha1(str(setting).encode()).hexdigest()[:6]
    filename = dir_path+'/{}-{}.json'.format(exp_name, hash_key)
    result.update(setting)
    with open(filename, 'w') as f:
        json.dump(result, f)

def sparse_to_tuple(sparse_mx):
    if not isinstance(sparse_mx, coo_matrix):
        sparse_mx = sparse_mx.tocoo()
    coords = np.vstack((sparse_mx.row, sparse_mx.col)).T  # (num_nonzero, 2)
    values = sparse_mx.data
    shape = sparse_mx.shape
    return coords, values, shape

class MultiHeadAttentionModule(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super(MultiHeadAttentionModule, self).__init__()
        # 定义多头注意力层
        self.multihead_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout,
                                                    batch_first=True)
        # 定义层归一化和全连接层
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        if isinstance(query, torch.sparse.Tensor):
            query = query.to_dense()

        if isinstance(key, torch.sparse.Tensor):
            key = key.to_dense()

        if isinstance(value, torch.sparse.Tensor):
            value = value.to_dense()

        # print(isinstance(value, torch.sparse.Tensor))

        # query, key, value 的维度应该是 [seq_len, batch_size, embed_dim]
        attn_output, attn_weights = self.multihead_attn(query, key, value, attn_mask=mask)
        # 跳跃连接 + 层归一化
        output = self.layer_norm(query + self.dropout(attn_output))
        # 输出经过全连接层
        output = self.fc(output)
        return output, attn_weights

def logging(msg, outdir, log_fpath):
    fpath = os.path.join(outdir, log_fpath)
    if not os.path.isdir(outdir):
        os.mkdir(outdir)
    with open(fpath, 'a') as fw:
        fw.write("%s\n" % msg)
    print(msg)

def log_learning_rates(optimizer, outdir, filename='lr.log'):
    lr_info = " | ".join(
        f"Group {i}: {group['lr']:.4e}" 
        for i, group in enumerate(optimizer.param_groups[:-1])  # 不包含loss_fn组
    )
    logging(f'LR after update: {lr_info}', outdir, filename)

def build_sample_features(big_dict, pathway_order):
    
    sample_features = {}
    samples = list(next(iter(big_dict.values())).keys())

    for sample in samples:
        features = []
        for pathway in pathway_order:
            features.append(big_dict[pathway][sample])
        sample_features[sample] = torch.stack(features)
    return sample_features

class TwoLForwardNetwork(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, dropout_pathway = 0.1,
                 num_class = 5,activation_func=nn.ReLU(),out_activation=None):
        super(TwoLForwardNetwork, self).__init__()
        
        self.layers = nn.Sequential(
             nn.Linear(input_dim, hidden_dim1).cuda(),
             nn.BatchNorm1d(hidden_dim1),
             #nn.LayerNorm(hidden_dim1),
             activation_func,
             nn.Dropout(dropout_pathway),
             nn.Linear(hidden_dim1, hidden_dim2).cuda(),
             nn.BatchNorm1d(hidden_dim2),
             #nn.LayerNorm(hidden_dim2),
             activation_func,
             nn.Dropout(dropout_pathway),
             nn.Linear(hidden_dim2, num_class)).cuda()
        
        self.out_activation = out_activation

    def forward(self, x):
        if self.out_activation:
            return self.out_activation(self.layers(x))
        else:
            return self.layers(x)

def split_dict(sample_dict, ratios=(0.8, 0.1, 0.1), seed=666):
    keys = list(sample_dict.keys())
    random.seed(seed)
    random.shuffle(keys)  # 打乱顺序
    
    n = len(keys)
    n_train = int(ratios[0] * n)
    n_val = int(ratios[1] * n)
    
    train_keys = keys[:n_train]
    val_keys = keys[n_train:n_train + n_val]
    test_keys = keys[n_train + n_val:]
    
    train_dict = {k: sample_dict[k] for k in train_keys}
    val_dict = {k: sample_dict[k] for k in val_keys}
    test_dict = {k: sample_dict[k] for k in test_keys}
    
    return train_dict, val_dict, test_dict

def adj_df_to_edge_index(adj_df: pd.DataFrame):
    """把DataFrame邻接矩阵转成edge_index"""
    adj = adj_df.values
    row, col = np.nonzero(adj)
    edge_index = torch.tensor([row, col], dtype=torch.long)
    return edge_index, adj_df.index.tolist()  # 同时返回基因名顺序

class ExpressionDataset(Dataset):
    def __init__(self, expr_df: pd.DataFrame, labels: dict):
        self.expr_df = expr_df
        self.labels = labels
        self.sample_names = list(labels.keys())

    def __len__(self):
        return len(self.sample_names)

    def __getitem__(self, idx):
        sample_name = self.sample_names[idx]
        label = self.labels[sample_name]
        return {
            'sample_name': sample_name,
            'label': torch.tensor(label, dtype=torch.long)
        }

def collate_fn(batch):
    sample_names = [item['sample_name'] for item in batch]
    labels = torch.stack([item['label'] for item in batch])
    return {
        'sample_names': sample_names,
        'labels': labels
    }

class GenePathwayProcessor:
    def __init__(self, pathways: dict):
        """预处理通路信息（仅需初始化一次）"""
        self.pathway_info = self._preprocess_pathways(pathways)
        # 提取所有通路的基因，构建全局基因索引（加速表达量查询）
        all_genes = list(set(g for p in self.pathway_info for g in p["genes"]))
        self.gene2idx = {g: i for i, g in enumerate(all_genes)}
        self.num_genes = len(all_genes)

    def _preprocess_pathways(self, pathways: dict):
        """将通路的邻接矩阵转换为边索引，并记录通路包含的基因"""
        pathway_list = []
        for p_name, adj_df in pathways.items():
            # 邻接矩阵→边索引（稀疏表示，节省内存）
            adj_matrix = adj_df.values  # 转为numpy矩阵
            edge_index, _ = dense_to_sparse(torch.tensor(adj_matrix, dtype=torch.float32))
            # 记录通路包含的基因（邻接矩阵的行/列名）
            pathway_genes = adj_df.index.tolist()
            pathway_list.append({
                "name": p_name,
                "edge_index": edge_index,  # [2, E]，E为边数
                "genes": pathway_genes,    # 通路包含的基因列表
                "num_genes": len(pathway_genes)
            })
        return pathway_list

    def process_batch(self, expr_df: pd.DataFrame, sample_names: list):

        num_samples = len(sample_names)
        
        # ----------------------
        # 1. 批量提取基因表达量（核心优化）
        # ----------------------
        # 构建样本×基因的表达矩阵（缺失值填0）
        expr_matrix = expr_df.reindex(self.gene2idx.keys())[sample_names].fillna(0.0).T
        expr_tensor = torch.tensor(expr_matrix.values, dtype=torch.float32)  # [N_samples, N_genes]
        
        # ----------------------
        # 2. 构建每个样本的通路网络数据
        # ----------------------
        batch_list = []
        for sample_idx in range(num_samples):
            sample_data_list = []
            # 提取当前样本的基因表达量（[N_genes,]）
            sample_expr = expr_tensor[sample_idx]
            
            for p in self.pathway_info:
                # 通路基因在全局基因索引中的位置
                gene_indices = torch.tensor([self.gene2idx[g] for g in p["genes"]], dtype=torch.long)
                # 提取当前通路的基因表达量（[num_genes_in_pathway, 1]）
                x = sample_expr[gene_indices].unsqueeze(1)  # 保持特征维度为1
                
                # 构建通路网络数据（表达量+边索引）
                data = Data(
                    x=x,
                    edge_index=p["edge_index"],  # 复用预计算的边索引
                    num_nodes=p["num_genes"]
                )
                sample_data_list.append(data)
            
            # 将当前样本的所有通路打包成一个Batch
            sample_batch = Batch.from_data_list(sample_data_list)
            batch_list.append(sample_batch)
        
        # ----------------------
        # 3. 合并所有样本的Batch（实现多样本并行）
        # ----------------------
        total_batch = Batch.from_data_list(batch_list)
        return total_batch

class InterpretableTransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_ff=128, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            dropout=dropout,
            embed_dim=d_model,
            num_heads=nhead,
            batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.ReLU(),
            nn.Linear(dim_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.self_attn(
            x_norm, x_norm, x_norm,
            need_weights=True,
            average_attn_weights=False
        )
        x = x + self.dropout(attn_out)
        x_norm = self.norm2(x)
        ffn_out = self.ffn(x_norm)
        x = x + self.dropout(ffn_out)
        return x, attn_weights
    
class PathwayGATModel(torch.nn.Module):
    def __init__(self,
                 gene_in_dim: int = 1,          
                 gene_hidden_dim1: int = 8,    
                 gene_hidden_dim2: int = 32,    
                 pathway_in_dim: int = 32,      
                 pathway_hidden_dim1: int = 8,  
                 MLP_input_dim: int = 2504,   
                 MLP_hidden_dim1: int = 512,
                 MLP_hidden_dim2: int = 64 , 
                 num_class: int = 5,            
                 dropout_pathway: float = 0.5,
                 GAT_dropout: float = 0.5,
                 transformer_heads: int = 2,
                 transformer_layers: int = 3,
                 final_pathway_dim: int =32):
        super(PathwayGATModel, self).__init__()

        # ----------------------
        # 通路内的基因图卷积(GCN)
        # ----------------------
        self.GCN1 = GCNConv(gene_in_dim, gene_hidden_dim1,add_self_loops=True).cuda()
        self.GCN2 = GCNConv(gene_hidden_dim1, gene_hidden_dim2,add_self_loops=True).cuda()
        self.gene_pool1 = PathwayAttention(gene_hidden_dim1)
        self.gene_pool2 = PathwayAttention(gene_hidden_dim2)
        # ----------------------
        # 通路GAT（处理通路间网络）
        # ----------------------
        self.pathway_GAT1 = GATConv(in_channels=pathway_in_dim,
                                    out_channels=pathway_hidden_dim1,
                                    heads=3,concat = False,dropout=GAT_dropout,
                                    add_self_loops= True,edge_dim=1)

        self.transformer_layers = nn.ModuleList([
            InterpretableTransformerLayer(
                dim_ff = 512,
                d_model = pathway_hidden_dim1,
                nhead = transformer_heads,
                dropout = 0.3
            ) for _ in range(transformer_layers)
        ])
        # ----------------------
        # 预测头（MLP）
        # ----------------------
        self.net = TwoLForwardNetwork(MLP_input_dim,
                                      MLP_hidden_dim1,
                                      MLP_hidden_dim2,
                                      dropout_pathway,num_class,
                                      activation_func = nn.GELU())
        
        self.activation = nn.GELU()
        self.gene_norm1 = GraphNorm(gene_hidden_dim1)
        self.gene_norm2 = GraphNorm(gene_hidden_dim2)
        #self.pathway_norm1 = GraphNorm(pathway_hidden_dim1)
        self.pre_transformer_proj = nn.Linear(pathway_hidden_dim1, pathway_hidden_dim1)
        self.pre_transformer_ln = nn.LayerNorm(pathway_hidden_dim1)
        self.token_dim_reduction = nn.Linear(gene_hidden_dim1+gene_hidden_dim2,final_pathway_dim)
        
    def forward(self,
            expr_df: pd.DataFrame,
            sample_names: list,
            pathway_edge_index1: torch.Tensor,
            pathway_edge_index2: torch.Tensor,
            pathway_edge_weight1: torch.Tensor,
            pathway_edge_weight2: torch.Tensor,
            batch_size: int) -> torch.Tensor:


        batch_res = processor.process_batch(expr_df, sample_names).cuda()

        gene_hidden1 = self.GCN1(batch_res.x, batch_res.edge_index)
        gene_hidden1 = self.gene_norm1(gene_hidden1, batch_res.batch)
        gene_hidden1 = self.activation(gene_hidden1)
        pathway_feats1,gene_atte1 = self.gene_pool1(gene_hidden1,batch_res.batch)
        #pathway_feats1 = global_mean_pool(gene_hidden1, batch_res.batch) 
        
        gene_hidden2 = self.GCN2(gene_hidden1, batch_res.edge_index)
        gene_hidden2 = self.gene_norm2(gene_hidden2, batch_res.batch)
        gene_hidden2 = self.activation(gene_hidden2)
        #pathway_feats2 = global_mean_pool(gene_hidden2, batch_res.batch)  
        pathway_feats2,gene_atte2 = self.gene_pool2(gene_hidden2,batch_res.batch)
        
        
        pathway_feats = torch.cat([pathway_feats1,pathway_feats2],dim=1)
        
        num_samples = len(sample_names)
        
        if num_samples == batch_size: 
            edge_index_b = pathway_edge_index1
            edge_weight_b = pathway_edge_weight1  
        else:       
            edge_index_b = pathway_edge_index2 
            edge_weight_b = pathway_edge_weight2 
            
        #pathway_batch = torch.arange(num_samples).repeat_interleave(258).cuda()
        
        p_emb1, (edge_idx_attn, gat_attn) = self.pathway_GAT1(
            pathway_feats,
            edge_index_b,
            edge_weight_b,
            return_attention_weights=True
        )
        # p_emb1 = self.pathway_norm1(p_emb1,pathway_batch)
        p_emb1 = self.activation(p_emb1)

        # 残差连接
        x = p_emb1 + pathway_feats
        x = x.view(num_samples, 258, -1)
        x = self.pre_transformer_ln(x)
        x = self.pre_transformer_proj(x)
        x_input = x
        transformer_attn_maps = []
        for layer in self.transformer_layers:
            x, attn = layer(x)
            transformer_attn_maps.append(attn)
        
        #p_emb1 = p_emb1.view(num_samples, 258, -1)
        final_feat = x_input+x
        #final_feat = x + p_emb1
        final_feat = self.token_dim_reduction(final_feat)
        
        # flatten 成样本向量
        final_feat = torch.flatten(final_feat, start_dim=1)
        out = self.net(final_feat)  #  [batch_size, num_class]

        return out,final_feat,gat_attn,edge_idx_attn,transformer_attn_maps,gene_atte1,gene_atte2
    
def tsne_visualize(X, y, perplexity=30, random_state=42, learning_rate=200,n_iter=1000,custom_colors=None,save_file=None):
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=random_state,
        n_iter=n_iter,
        learning_rate=learning_rate
    )
    X_tsne = tsne.fit_transform(X)
    
    plt.figure(figsize=(10, 8))  # 放大画布，避免标签重叠
    unique_labels = np.unique(y)
    n_classes = len(unique_labels)
    
    if custom_colors is None:
        colors = plt.cm.get_cmap("tab20b")(np.linspace(0, 1, n_classes))
    else:
        if len(custom_colors) < n_classes:
            raise ValueError(f"自定义颜色数量({len(custom_colors)})少于类别数({n_classes})")
        colors = custom_colors[:n_classes]
    
    for i, label in enumerate(unique_labels):
        mask = y == label
        # 绘制散点
        plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=[colors[i]], label=label, s=10, alpha=0.8)
        
        # 核心：在每簇中心显示类别标签
        # 计算当前类别所有点的坐标均值（簇中心）
        cluster_center = np.mean(X_tsne[mask], axis=0)
        # 绘制类别文本（避免遮挡，可微调xytext偏移）
        plt.annotate(
            label,  # 要显示的类别名
            xy=cluster_center,  # 文本位置（簇中心）
            xytext=(5, 5),  # 文本偏移量（避免和点重叠）
            textcoords="offset points",
            fontsize=8,  # 字体大小
            fontweight="bold",  # 加粗
            bbox=dict(boxstyle="round,pad=0.3", fc=colors[i], alpha=0.5)  # 背景框（增强可读性）
        )
    
    plt.title("t-SNE Visualization")
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")  # 图例放右侧，避免遮挡
    plt.tight_layout()
    plt.savefig(save_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return X_tsne

def reconstruct_attention_matrix(all_edge_index, all_att_weights, num_nodes_per_graph):
    all_batches_results = []

    for edge_index, att in zip(all_edge_index, all_att_weights):
        # 确保 edge_index 是 long 类型，att 是 float 类型
        if isinstance(edge_index, np.ndarray):
            edge_index = torch.from_numpy(edge_index).long()
        if isinstance(att, np.ndarray):
            att = torch.from_numpy(att).float()
        
        if att.dim() > 1:
            # att :[E, heads]，对头取均值
            att = att.mean(dim=-1)

        # 2. 推断样本数量
        max_node_idx = edge_index.max().item()
        num_samples_in_batch = int(max_node_idx // num_nodes_per_graph) + 1

        # 3. 计算每条边属于哪个样本 (整除)
        sample_ids = edge_index[0] // num_nodes_per_graph
        
        # 4. 获取目标节点在样本内的局部索引 (取余)
        col_local = edge_index[1] % num_nodes_per_graph

        # 5. 构建组合索引
        combined_idx = sample_ids * num_nodes_per_graph + col_local

        # 6. 聚合计算
        out_size = num_samples_in_batch * num_nodes_per_graph
        combined_idx = combined_idx.long() 

        flat_result = scatter_mean(att, combined_idx, dim=0, dim_size=out_size)

        # 7. Reshape 并存入 list
        batch_matrix = flat_result.view(num_samples_in_batch, num_nodes_per_graph)
        all_batches_results.append(batch_matrix)

    # 8. 合并所有批次
    return torch.cat(all_batches_results, dim=0)

class PathwayAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )

    def forward(self, x, batch):
        """
        Args:
            x: [Total_Genes_in_Batch, dim]
            batch: [Total_Genes_in_Batch] 记录每个基因属于哪个通路
        """
        # 1. 计算每个基因的原始得分
        raw_attn = self.gate(x)          # [Total_Genes_in_Batch, 1]
        
        # 2. 【关键】分段 Softmax
        # 它会确保同一个通路内的基因权重之和为 1，不同通路互不影响
        attn = softmax(raw_attn, batch)  # [Total_Genes_in_Batch, 1]
        
        # 3. 加权特征
        weighted_x = x * attn            # [Total_Genes_in_Batch, dim]
        
        # 4. 聚合得到通路特征
        # 相当于 global_sum_pool，但因为权重和为 1，本质上是加权平均
        pathway_feats = scatter_sum(weighted_x, batch, dim=0) 
        
        return pathway_feats,attn
    

In [ ]:
# ====== Argument Parsing ====== #
parser = argparse.ArgumentParser()
parser.add_argument('--WORKDIR_PATH', type=str, default="/home/nanyuan/hdd/pathway_chat")
parser.add_argument('--model_name', type=str, default='my')
parser.add_argument('--outdir',type=str,default="/home/nanyuan/hdd/pathway_chat/res/test_temp")

# === Train setting === #
parser.add_argument('--learning_rate', type=float, default=1e-4)
parser.add_argument('--epochs', type=int, default=100)
parser.add_argument('--batch_size', type=int, default=10)
parser.add_argument('--weight_decay', type=float, default=0)
parser.add_argument('--patience', type=int, default=10)
parser.add_argument('--testset_yes', type=bool, default=True)
args = parser.parse_known_args()[0]

In [ ]:
with open("/home/nanyuan/hdd/pathway_chat/data/pathways_adjacency(20).pkl", "rb") as f:  # 注意要用 "rb" 读二进制
    pathways_matrix = pickle.load(f)
pathway_adj = pd.read_csv('/home/nanyuan/hdd/pathway_chat/data/pathway_weight_adj(20).csv',index_col=0) 
exp_data = pd.read_pickle('/home/nanyuan/hdd/pathway_chat/data/TCGA/cancer_integrate1/primary_exp.pkl')
label_data = pd.read_csv("/home/nanyuan/hdd/pathway_chat/data/TCGA/cancer_integrate1/primary_label.csv")
pathway_order = list(pathways_matrix.keys())
pathway_adj = pathway_adj.loc[pathway_order, :]
pathway_adj = pathway_adj.loc[:, pathway_order]
#pathway_adj = pathway_adj.where(pathway_adj>= 0.5,0)
sparseTensor = torch.tensor(pathway_adj.values).to_sparse().cuda()
pathway_ind = sparseTensor.indices()
pathway_weight = sparseTensor.values()

processor = GenePathwayProcessor(pathways_matrix)

In [ ]:
model_max = torch.load('/home/nanyuan/hdd/pathway_chat/res/test_temp/fold_2_best.model',weights_only=False)
counts = torch.tensor(label_data['label'].value_counts().sort_index().values, dtype=torch.float).cuda()
n_classes = len(counts)
total = counts.sum()
class_weights = total / (n_classes * counts).cuda()
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
val_samples = np.load("/home/nanyuan/hdd/pathway_chat/res/test_temp/fold_2_val_samples.npy")
val_exp = exp_data#[val_samples]
val_label = label_data#[label_data['samples'].isin(val_samples)]


edge_index_b = []
for i in range(args.batch_size):
    offset = i * 258
    edge_index_b.append(pathway_ind + offset)
edge_index_b = torch.cat(edge_index_b, dim=1)
edge_weight_b = pathway_weight.repeat(args.batch_size).float()

label_dict  = dict(zip(val_label['samples'], val_label['label']))
val_dataset = ExpressionDataset(val_exp, label_dict)
val_dataloader = DataLoader(val_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)
other_val = val_exp.shape[1] % args.batch_size
edge_index_b_val = []
for i in range(other_val): 
    offset = i * 258 
    edge_index_b_val.append(pathway_ind + offset) 

edge_index_b_val = torch.cat(edge_index_b_val, dim=1) 
edge_weight_b_val = pathway_weight.repeat(other_val)
edge_weight_b_val = edge_weight_b_val.float()


In [ ]:
#注意力权重提取

def test(model,test_exp, test_loader, edge_index_b,edge_index_b_test, edge_weight_b,edge_weight_b_test,batchsize,loss_fn):
    model.eval() 
    with torch.no_grad():  #####禁用梯度计算
        # ====== Test ====== #
        list_test_loss  = []
        list_test_out = []
        list_test_true = []
        list_test_sample = []
        list_test_Gat_atten = []
        list_test_Gat_index = []
        list_test_Tran_atten = []
        list_test_Gene_atten1 = []
        list_test_Gene_atten2 = []

        for batch in tqdm(test_loader):
            sample_names = batch['sample_names']  # list[str]
            test_label = batch['labels'].cuda()
            
            sample_pred,_,Gat_atten,edge_idx_attn,transformer_attn,gene_att1,gene_att2= model(test_exp, sample_names, edge_index_b,edge_index_b_test, edge_weight_b.unsqueeze(1),edge_weight_b_test.unsqueeze(1),batchsize)
            output_loss = loss_fn(sample_pred, test_label)

            list_test_sample.append(sample_names)
            list_test_out.append(sample_pred.detach().cpu().numpy())
            list_test_loss.append(output_loss.detach().cpu().numpy())
            list_test_true.append(test_label.detach().cpu().numpy())
            list_test_Gat_atten.append(Gat_atten.detach().cpu().numpy())
            list_test_Gat_index.append(edge_idx_attn.detach().cpu().numpy())
            list_test_Tran_atten.append(transformer_attn)
            list_test_Gene_atten1.append(gene_att1)
            list_test_Gene_atten2.append(gene_att2)
            
            
    return  list_test_sample,list_test_Gat_atten,list_test_Gat_index,list_test_Tran_atten,list_test_Gene_atten1,list_test_Gene_atten2


val_samples = np.load("/home/nanyuan/hdd/pathway_chat/res/test_temp/fold_2_val_samples.npy")
val_exp = exp_data[val_samples]
val_label = label_data[label_data['samples'].isin(val_samples)]
label_dict  = dict(zip(val_label['samples'], val_label['label']))
val_dataset = ExpressionDataset(val_exp, label_dict)
val_dataloader = DataLoader(val_dataset,batch_size=args.batch_size,shuffle=False,collate_fn=collate_fn)

edge_index_b = []
for i in range(args.batch_size):
    offset = i * 258
    edge_index_b.append(pathway_ind + offset)
edge_index_b = torch.cat(edge_index_b, dim=1)
edge_weight_b = pathway_weight.repeat(args.batch_size).float()

other_val = val_exp.shape[1] % args.batch_size
edge_index_b_val = []
for i in range(other_val): 
    offset = i * 258 
    edge_index_b_val.append(pathway_ind + offset) 

edge_index_b_val = torch.cat(edge_index_b_val, dim=1) 
edge_weight_b_val = pathway_weight.repeat(other_val)
edge_weight_b_val = edge_weight_b_val.float()
list_val_sample,list_val_Gat_atten,list_val_Gat_index,list_val_Tran_atten,list_test_Gene_atten1,list_test_Gene_atten2 = test(
    model_max,val_exp, val_dataloader, edge_index_b,edge_index_b_val, edge_weight_b,edge_weight_b_val,args.batch_size,loss_fn)


In [ ]:
#GAT注意力整合----
num_pathways = 258
pathway_names = pathway_order
all_batch_dfs = []

# 批次迭代处理
for i in range(len(list_val_Gat_atten)):
    # 提取当前 Batch 数据
    gat_attn = list_val_Gat_atten[i]
    edge_idx = list_val_Gat_index[i]
    sample_names = list_val_sample[i]
    num_samples = len(sample_names)
    
    # 确保是 Tensor 格式
    if not isinstance(gat_attn, torch.Tensor):
        gat_attn = torch.tensor(gat_attn)
    if not isinstance(edge_idx, torch.Tensor):
        edge_idx = torch.tensor(edge_idx)

    # 计算多头均值并确定源节点
    avg_gat_attn = gat_attn.mean(dim=1) 
    source_nodes = edge_idx[0] 
    
    # 计算列和 (Column Sum)
    # 这里的 dim_size 必须严格等于当前 batch 的 样本数 * 258
    col_sum = scatter_sum(
        avg_gat_attn, 
        source_nodes, 
        dim=0, 
        dim_size=num_samples * num_pathways
    )
    
    # 计算列均值：分母为通路数，因为每个样本有一个通路的注意力矩阵
    col_mean = col_sum / num_pathways
    
    # 重构为矩阵并转为 DataFrame
    batch_matrix = col_mean.view(num_samples, num_pathways).cpu().numpy()
    batch_df = pd.DataFrame(
        batch_matrix,
        index=sample_names,
        columns=pathway_names
    )
    
    all_batch_dfs.append(batch_df)

# --- 3. 合并所有批次 ---
df_final_importance = pd.concat(all_batch_dfs, axis=0)

print(df_final_importance.mean(axis=0).sort_values(ascending=False).head(20))
#df_final_importance.to_csv('/home/nanyuan/hdd/pathway_chat/Figures/Figure 3/GAT_pathway_atten.csv')

In [ ]:
#Transformer注意力整合----
num_pathways = 258
pathway_names = pathway_order
all_trans_dfs = []


for b_idx in range(len(list_val_Tran_atten)):

    layers_list = list_val_Tran_atten[b_idx] 
    #对层取平均
    mean_layer_attn = torch.stack(layers_list).mean(dim=0)

    # 整合多头 
    avg_head_attn = mean_layer_attn.mean(dim=1)
    
    # 4.单个样本中所有通路的活性均值
    trans_col_mean = avg_head_attn.mean(dim=1)
    
    # 5. 绑定样本名并转为 DataFrame
    sample_names = list_val_sample[b_idx]
    batch_df = pd.DataFrame(
        trans_col_mean.detach().cpu().numpy(),
        index=sample_names,
        columns=pathway_names
    )
    
    all_trans_dfs.append(batch_df)

# 合并所有批次 ---
df_trans_final = pd.concat(all_trans_dfs, axis=0)

print(df_trans_final.mean(axis=0).sort_values(ascending=False).head(20))
#df_trans_final.to_csv('/home/nanyuan/hdd/pathway_chat/Figures/Figure 3/Trans_pathway_atten.csv')

In [ ]:
#基因权重提取----
all_sample_gene_importance = {} # 用于存储最终结果 {样本名: {通路名: 权重数组}}

model_max.eval()
with torch.no_grad():
    for batch in tqdm(val_dataloader):
        sample_names = batch['sample_names']  # list[str]
        test_label = batch['labels'].cuda() # 假设你的表达量数据
        
        sample_pred,_,Gat_atten,edge_idx_attn,transformer_attn,gene_att1,gene_att2 = model_max(val_exp,sample_names,edge_index_b,
                                                                                              edge_index_b_val, edge_weight_b,
                                                                                              edge_weight_b_val,args.batch_size)
        raw_attn1 = gene_att1 
        raw_attn2 = gene_att2
        
        temp_total_batch = processor.process_batch(val_exp, sample_names)
        batch_idx = temp_total_batch.batch.to(raw_attn1.device) 
        
        gene_importance1 = softmax(raw_attn1, batch_idx) # [Total_Genes_in_Batch, 1]
        gene_importance2 = softmax(raw_attn2, batch_idx) 
        
        gene_importance_np1 = gene_importance1.flatten().cpu().detach().numpy()
        gene_importance_np2 = gene_importance2.flatten().cpu().detach().numpy()
        batch_idx_np = batch_idx.cpu().numpy()

        # 4. 结构化拆解：利用 batch 索引的连续性
        num_pathways = len(processor.pathway_info)
        for s_idx, s_name in enumerate(sample_names):
            all_sample_gene_importance[s_name] = {}
            
            for p_idx, p_info in enumerate(processor.pathway_info):
                # 计算当前样本中，当前通路的全局唯一 ID
                instance_id = s_idx * num_pathways + p_idx
                
                # 提取属于该通路的权重
                mask = (batch_idx_np == instance_id)
                p_weights1 = gene_importance_np1[mask]
                p_weights2 = gene_importance_np2[mask]
                p_weights = (p_weights1+p_weights2)/2
                # 对应基因名并保存
                # p_info["genes"] 是你在处理器中存好的原始基因顺序
                all_sample_gene_importance[s_name][p_info["name"]] = dict(zip(p_info["genes"], p_weights))


first_sample = sample_names[0]
first_pathway = processor.pathway_info[0]["name"]
print(f"样本: {first_sample} | 通路: {first_pathway}")
print(all_sample_gene_importance[first_sample][first_pathway])
with open("/home/nanyuan/hdd/pathway_chat/Figures/Figure 3/gene_importance.pkl", "wb") as f:
    pickle.dump(all_sample_gene_importance, f)

In [ ]:
with open("/home/nanyuan/hdd/pathway_chat/Figures/Figure 3/gene_importance.pkl", "rb") as f:
    gene_importance = pickle.load(f)

target_pathway = "Rap1 signaling pathway" 

# {样本1: {基因A: 值, 基因B: 值}, 样本2: {基因A: 值...}}
pathway_data = {
    sample: genes 
    for sample, pathways in gene_importance.items() 
    if target_pathway in pathways and (genes := pathways[target_pathway])
}

# 转换为 DataFrame
df_pathway = pd.DataFrame.from_dict(pathway_data, orient='index')

df_final = df_pathway.T
df_final.to_csv('/home/nanyuan/hdd/pathway_chat/Figures/Figure 3/Trans_Rap1 signaling.csv')